# Análise de Dados - TSE 2022 vs Câmara dos Deputados

Este notebook analisa o matching entre os dados do TSE (Tribunal Superior Eleitoral) e a API da Câmara dos Deputados.

In [ ]:
import json
import pandas as pd
from collections import Counter
import unicodedata

def normalize_name(name):
    """Normaliza nome para matching"""
    if not name:
        return ""
    n = name.upper()
    n = unicodedata.normalize('NFD', n)
    n = ''.join(c for c in n if c not in '\u0300\u0301\u0302\u0303\u0304\u0305\u0306\u0307\u0308\u0309\u030a\u030b\u030c\u030d\u030e\u030f\u0310\u0311\u0312\u0313\u0314\u0315\u0316\u0317\u0318\u0319\u031a\u031b\u031c\u031d\u031e\u031f\u0320\u0321\u0322\u0323\u0324\u0325\u0326\u0327\u0328\u0329\u032a\u032b\u032c\u032d\u032e\u032f\u0330\u0331\u0332\u0333')
    n = ''.join(c for c in n if c.isalpha() or c == ' ')
    return n.strip()

## 1. Carregar dados do TSE

In [ ]:
# Carregar dados TSE 2022
with open('public/data/tse-deputados-2022.json', 'r', encoding='utf-8') as f:
    tse_data = json.load(f)

print(f"Total de entradas no TSE: {len(tse_data)}")

# Converter para DataFrame
tse_df = pd.DataFrame([
    {'nome': nome, **dados} for nome, dados in tse_data.items()
])
print(f"\nDataFrame shape: {tse_df.shape}")
tse_df.head(10)

## 2. Estatísticas de Raça no TSE

In [ ]:
# Contagem por raça
raca_counts = tse_df['raca'].value_counts()
print("Distribuição de raça no TSE:")
print(raca_counts)
print(f"\nTotal: {raca_counts.sum()}")

In [ ]:
# Gráfico de pizza
import matplotlib.pyplot as plt

raca_counts.plot(kind='pie', autopct='%1.1f%%', figsize=(8, 8))
plt.title('Distribuição de Raça - TSE 2022')
plt.ylabel('')
plt.show()

## 3. Carregar dados da Câmara

In [ ]:
# Carregar dados da Câmara
with open('public/data/chamber_raw.json', 'r', encoding='utf-8') as f:
    chamber_data = json.load(f)

print(f"Dados da Câmara: {list(chamber_data.keys())}")

# Unir deputados e senadores
parlamentares = chamber_data.get('deputados', []) + chamber_data.get('senadores', [])
print(f"Total de parlamentares: {len(parlamentares)}")

## 4. Análise de nomes

In [ ]:
# Verificar estrutura dos dados
if parliamentares:
    print("Exemplo de parlamentar:")
    print(json.dumps(parlamentares[0], indent=2, ensure_ascii=False)[:1000])

## 5. Matching de nomes

In [ ]:
# Criar dicionário TSE normalizado
tse_normalized = {normalize_name(nome): dados for nome, dados in tse_data.items()}

matched = []
unmatched = []

for p in parliamentares:
    # Usar nomeUrna ou nome
    nome_urna = p.get('nomeUrna') or p.get('nome') or ''
    normalized = normalize_name(nome_urna)
    
    if normalized in tse_normalized:
        matched.append({
            'nome': nome_urna,
            'normalized': normalized,
            'tse_raca': tse_normalized[normalized]['raca'],
            'tse_genero': tse_normalized[normalized]['genero']
        })
    else:
        unmatched.append({
            'nome': nome_urna,
            'normalized': normalized,
            'partido': p.get('partido'),
            'uf': p.get('uf')
        })

print(f"Matched: {len(matched)}")
print(f"Unmatched: {len(unmatched)}")
print(f"Taxa de match: {100*len(matched)/(len(matched)+len(unmatched)):.1f}%")

## 6. Análise dos não correspondidos

In [ ]:
unmatched_df = pd.DataFrame(unmatched)
print("Primeiros 30 nomes não correspondidos:")
print(unmatched_df[['nome', 'partido', 'uf']].head(30).to_string())

## 7. Tentativa de matching flexível

In [ ]:
# Tentar matching por primeiro + último nome
flexible_matches = []

for p in unmatched[:100]:  # Limitar para performance
    parts = p['normalized'].split()
    if len(parts) >= 2:
        first_last = parts[0] + ' ' + parts[-1]
        for tse_name, tse_dados in tse_data.items():
            tse_norm = normalize_name(tse_name)
            if first_last in tse_norm or tse_norm in first_last:
                flexible_matches.append({
                    'parlamentar': p['nome'],
                    'tse_name': tse_name,
                    'raca': tse_dados['raca'],
                    'genero': tse_dados['genero']
                })
                break

print(f"Matches flexíveis encontrados: {len(flexible_matches)}")
print("\nExemplos:")
for m in flexible_matches[:10]:
    print(f"  {m['parlamentar']} <-> {m['tse_name']} -> {m['raca']}")

## 8. Distribuição final de raça

In [ ]:
# Distribuição de raça nos parliamentares
matched_df = pd.DataFrame(matched)
print("Distribuição de raça nos parlamentar matched:")
print(matched_df['tse_raca'].value_counts())

# Adicionar aos não matched (ficarão sem raça)
total_com_raca = len(matched)
total_sem_raca = len(unmatched)
total = total_com_raca + total_sem_raca

print(f"\n--- Resumo ---")
print(f"Total parlamentares: {total}")
print(f"Com raça (TSE): {total_com_raca} ({100*total_com_raca/total:.1f}%)")
print(f"Sem raça: {total_sem_raca} ({100*total_sem_raca/total:.1f}%)")

In [ ]:
# Gráfico final
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição de raça
matched_df['tse_raca'].value_counts().plot(kind='bar', ax=axes[0], color=['blue', 'pink', 'green', 'orange', 'purple'])
axes[0].set_title('Raça - Parlamentares Matched')
axes[0].set_ylabel('Count')

# Com vs Sem
axes[1].pie([total_com_raca, total_sem_raca], labels=['Com raça (TSE)', 'Sem raça'], 
            autopct='%1.1f%%', colors=['#22c55e', '#ef4444'])
axes[1].set_title('Cobertura de Raça')

plt.tight_layout()
plt.show()